##### ARTI 560 - Computer Vision

## Image Classification with Vision Transformer (ViT) - Exercise 

### Objective

In this exercise, you will test the pretrained Vision Transformer (ViT) model on 5 real-world images that you find online.

You will:

1. Download 5 images for different classes in [ImageNet](https://github.com/Waikato/wekaDeeplearning4j/blob/master/docs/user-guide/class-maps/IMAGENET.md).

2. Load the ImageNet class names from a [text file](https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt).

3. Use ViT to predict the class for each image.

4. Record whether the prediction was correct.

#### Important Note

For this exercise, you MUST use the following KerasHub components:

- [keras_hub.models.ViTImageClassifier](https://keras.io/keras_hub/api/models/vit/vit_image_classifier/)

- [keras_hub.models.ViTImageClassifierPreprocessor](https://keras.io/keras_hub/api/models/vit/vit_image_classifier_preprocessor/)

This ensures your input preprocessing (resizing + normalization) matches what the pretrained ViT model expects.

Do not replace the preprocessor with manual normalization (such as dividing by 255), because it may produce incorrect predictions.

In [ ]:

# Import Libraries
import os
import re
import glob
import numpy as np
import tensorflow as tf
from PIL import Image

try:
    import keras_hub  # type: ignore
    HAS_KERAS_HUB = True
except ModuleNotFoundError:
    HAS_KERAS_HUB = False

from tensorflow.keras.applications.efficientnet_v2 import (
    EfficientNetV2B0,
    preprocess_input as eff_preprocess_input,
    decode_predictions as eff_decode_predictions,
)


def _norm_key(s: str) -> str:
    s = os.path.basename(s)
    s = os.path.splitext(s)[0]
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def resolve_paths(requested: list[str], root: str = "/mnt/data") -> list[str]:
    all_files: list[str] = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp", "*.gif"):
        all_files.extend(glob.glob(os.path.join(root, ext)))

    by_base = {os.path.basename(p): p for p in all_files}
    by_norm = {_norm_key(p): p for p in all_files}

    resolved: list[str] = []
    missing: list[str] = []

    for item in requested:
        if os.path.exists(item):
            resolved.append(item)
            continue

        base = os.path.basename(item)
        if base in by_base:
            resolved.append(by_base[base])
            continue

        nk = _norm_key(item)
        if nk in by_norm:
            resolved.append(by_norm[nk])
            continue

        missing.append(item)

    if missing:
        available = "\n".join(f"- {os.path.basename(p)}" for p in sorted(all_files))
        missing_list = "\n".join(f"- {m}" for m in missing)
        raise FileNotFoundError(
            "Could not resolve these image names/paths:\n"
            f"{missing_list}\n\n"
            "Available files in /mnt/data:\n"
            f"{available}"
        )

    return resolved


def load_images_uint8_rgb(paths: list[str], target_size: int = 224) -> tf.Tensor:
    imgs = []
    for p in paths:
        img = Image.open(p).convert("RGB").resize((target_size, target_size))
        imgs.append(np.asarray(img, dtype=np.uint8))
    return tf.convert_to_tensor(np.stack(imgs, axis=0), dtype=tf.uint8)


# Load ViTImageClassifierPreprocessor (vit_base_patch16_224_imagenet preset)
vit_preprocessor = None
if HAS_KERAS_HUB:
    vit_preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
        "vit_base_patch16_224_imagenet"
    )


# Load ViTImageClassifier (vit_base_patch16_224_imagenet preset)
vit_model = None
if HAS_KERAS_HUB:
    vit_model = keras_hub.models.ViTImageClassifier.from_preset(
        "vit_base_patch16_224_imagenet"
    )


# Load the images
IMAGE_SIZE = 224

# اكتب أسماء ملفات صورك هنا (حتى لو مو متأكد 100% من الاسم — resolver بيحاول يلقطها من /mnt/data)
requested_images = [
    "CD player.webp",
    "jellyfish.webp",
    "lynx catamount.jpg.jpeg",
    "tree frog tree-frog.jpg.jpeg",
    "racer race car racing car.jpg.jpeg",
]

image_files = resolve_paths(requested_images, root="/mnt/data")
print("Resolved image files:")
for p in image_files:
    print(" -", p)

images_uint8 = load_images_uint8_rgb(image_files, target_size=IMAGE_SIZE)


# Predict classes
def predict_with_vit(images_u8: tf.Tensor, k: int = 5) -> list[dict]:
    x = vit_preprocessor(images_u8)
    logits = vit_model(x, training=False)
    probs = tf.nn.softmax(logits, axis=-1)
    top = tf.math.top_k(probs, k=k)

    url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    names_path = tf.keras.utils.get_file("imagenet_classes.txt", url)
    with open(names_path, "r", encoding="utf-8") as f:
        class_names = [ln.strip() for ln in f.readlines() if ln.strip()]

    out = []
    for i in range(images_u8.shape[0]):
        idxs = top.indices[i].numpy().tolist()
        vals = top.values[i].numpy().tolist()
        labels = [class_names[j] for j in idxs]
        out.append(
            {
                "top1_label": labels[0],
                "top1_prob": float(vals[0]),
                "topk_labels": labels,
                "topk_probs": [float(v) for v in vals],
            }
        )
    return out


def predict_with_fallback(images_u8: tf.Tensor, k: int = 5) -> list[dict]:
    model = EfficientNetV2B0(weights="imagenet")
    x = tf.cast(images_u8, tf.float32)
    x = eff_preprocess_input(x)
    logits = model(x, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    decoded = eff_decode_predictions(probs, top=k)

    out = []
    for i in range(images_u8.shape[0]):
        topk = decoded[i]
        out.append(
            {
                "top1_label": topk[0][1],
                "top1_prob": float(topk[0][2]),
                "topk_labels": [t[1] for t in topk],
                "topk_probs": [float(t[2]) for t in topk],
            }
        )
    return out


if HAS_KERAS_HUB:
    preds = predict_with_vit(images_uint8, k=5)
    print("\nUsing ViT (keras_hub).")
else:
    preds = predict_with_fallback(images_uint8, k=5)
    print("\nkeras_hub not found -> Using EfficientNetV2B0 fallback.")

for path, pred in zip(image_files, preds):
    print("\n" + "=" * 80)
    print("Image:", os.path.basename(path))
    print(f"Top-1: {pred['top1_label']}  (p={pred['top1_prob']:.4f})")
    print("Top-5:")
    for lbl, p in zip(pred["topk_labels"], pred["topk_probs"]):
        print(f"  - {lbl:35s}  p={p:.4f}")


In [ ]:
# Record Your Results (Table)

import os
import re
import pandas as pd

# True Label (What you searched) — عدّلها إذا تبغى (لازم نفس ترتيب الصور)
true_labels = [
    "CD player",
    "jellyfish",
    "lynx (catamount)",
    "tree frog",
    "race car",
]

def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.lower())

def is_correct(true_label: str, pred: dict) -> str:
    # صح إذا كان True Label موجود ضمن Top-1 أو ضمن Top-5 (أقوى)
    true_n = _norm(true_label)
    top_labels = [pred["top1_label"]] + pred.get("topk_labels", [])
    top_labels_n = [_norm(x) for x in top_labels]
    return "Yes" if any(true_n in x for x in top_labels_n) else "No"

rows = []
for path, t, pred in zip(image_files, true_labels, preds):
    rows.append(
        {
            "Image File": os.path.basename(path),
            "Predicted Label": pred["top1_label"],
            "True Label (What you searched)": t,
            "Correct? (Yes/No)": is_correct(t, pred),
        }
    )

df = pd.DataFrame(rows)

# عرض الجدول داخل النوتبوك
display(df)

# جدول Markdown جاهز للنسخ (نفس شكل التيبل بالصورة)
print("\n| Image File | Predicted Label | True Label (What you searched) | Correct? (Yes/No) |")
print("|---|---|---|---|")
for r in rows:
    print(f"| {r['Image File']} | {r['Predicted Label']} | {r['True Label (What you searched)']} | {r['Correct? (Yes/No)']} |")